# 5. Download PDFs

This notebook starts from the cleaned literature table from notebook 3/4 and downloads available open-access PDFs into year-specific `raw_data/publications_<year>` folders.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import ast
import os
import re
import sys

import pandas as pd

# =========================================================
# PATH CONFIG
# =========================================================
# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

FUNCTIONS_DIR = DATASET_CURATION_CODE_ROOT / "functions"
if not FUNCTIONS_DIR.exists():
    raise FileNotFoundError(
        f"Functions folder not found: {FUNCTIONS_DIR}. "
        "Copy code/dataset_curation/functions to the OneDrive code root or adjust CODE_ROOT."
    )

if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))

from download_oa_pdfs import download_pdfs_by_doi

SCHOLARLY_FILE = DATA_ROOT / "processed_data" / "scholarly_final_reduced.xlsx"
REPORT_DIR = DATA_ROOT / "processed_data"
PDF_ROOT = DATA_ROOT / "raw_data"


## Configure Year And Load Literature Table

Set `YEAR` and optional `ROW_LIMIT` before running the download cells.


In [ ]:
YEAR = 2010
ROW_LIMIT = 30000  # set to None for all rows in the selected year

report_file = REPORT_DIR / f"report_{YEAR}.csv"
output_dir = PDF_ROOT / f"publications_{YEAR}"
output_dir.mkdir(parents=True, exist_ok=True)

df_scholarly = pd.read_excel(SCHOLARLY_FILE)
print(f"Loaded {len(df_scholarly):,} literature rows from {SCHOLARLY_FILE}")
print(f"PDF output folder: {output_dir}")
print(f"Report file: {report_file}")


## Build Download Queue

Rows already present in the report are skipped, so the notebook can resume after interrupted runs.


In [ ]:
if report_file.exists():
    df_report_old = pd.read_csv(report_file, index_col=0).fillna("None")
    processed_pubs = df_report_old["doi"].dropna().tolist()
else:
    df_report_old = pd.DataFrame(
        columns=["doi", "saved_filename", "status", "winning_url", "explanation"]
    ).fillna("None")
    processed_pubs = []

year_rows = df_scholarly[df_scholarly["dates"] == YEAR]
if ROW_LIMIT is not None:
    year_rows = year_rows.iloc[:ROW_LIMIT]

doi_url_dict_all = {}
doi_url_dict_download = {}

for row in year_rows.index.to_list():
    doi = df_scholarly.loc[row, "doi"]
    if pd.isna(doi):
        continue

    if pd.isna(df_scholarly.loc[row, "open_access_pdf_urls"]):
        url_list = "unknown"
    else:
        url_list = ast.literal_eval(df_scholarly.loc[row, "open_access_pdf_urls"])

    doi_url_dict_all[doi] = url_list
    if doi not in processed_pubs:
        doi_url_dict_download[doi] = url_list

print(f"Rows for selected year: {len(year_rows):,}")
print(f"Total DOI/url entries: {len(doi_url_dict_all):,}")
print(f"Remaining to download: {len(doi_url_dict_download):,}")


## Download PDFs

This cell performs the network download and updates the report for the selected year.


In [ ]:
report = download_pdfs_by_doi(
    doi_url_dict_download,
    output_dir=str(output_dir),
    max_workers=16,
    timeout=30,
)

df_report_new = pd.DataFrame(
    report,
    columns=["doi", "saved_filename", "status", "winning_url", "explanation"],
)

df_report = pd.merge(df_report_old, df_report_new, how="outer")
df_report.to_csv(report_file)

print(f"Saved report: {report_file}")
print(df_report["status"].value_counts(dropna=False))


## Optional Duplicate PDF Cleanup

The cleanup is disabled by default. First run with `DELETE_DUPLICATES = False` to preview duplicate filenames, then set it to `True` only when the preview looks right.


In [ ]:
DELETE_DUPLICATES = False

pdf_folders = sorted(PDF_ROOT.glob("publications_*"))

def get_base_name(filename):
    name = os.path.splitext(filename)[0]
    return re.sub(r" \(\d+\)$", "", name)

for pdf_folder in pdf_folders:
    print(f"\n--- Processing folder: {pdf_folder} ---")
    all_pdf_files = [f.name for f in pdf_folder.iterdir() if f.suffix.lower() == ".pdf"]

    base_to_keep = {}
    files_to_delete = []

    for filename in sorted(all_pdf_files):
        base_name = get_base_name(filename)

        if base_name not in base_to_keep:
            base_to_keep[base_name] = filename
            continue

        current_best = base_to_keep[base_name]
        current_has_suffix = bool(re.search(r" \(\d+\)\.pdf$", current_best, re.IGNORECASE))
        new_has_suffix = bool(re.search(r" \(\d+\)\.pdf$", filename, re.IGNORECASE))

        # Prefer the clean filename over a duplicate suffix.
        if current_has_suffix and not new_has_suffix:
            files_to_delete.append(current_best)
            base_to_keep[base_name] = filename
        else:
            files_to_delete.append(filename)

    print(f"Total PDFs: {len(all_pdf_files)}")
    print(f"Unique PDFs: {len(base_to_keep)}")
    print(f"Duplicates identified: {len(files_to_delete)}")
    print("Examples:", files_to_delete[:5])

    if DELETE_DUPLICATES:
        for filename in files_to_delete:
            file_path = pdf_folder / filename
            try:
                file_path.unlink()
            except Exception as e:
                print(f"Failed to delete {filename}: {e}")

print("\nDuplicate cleanup preview complete." if not DELETE_DUPLICATES else "\nDuplicate cleanup complete.")


## Count PDFs By Year


In [ ]:
total_pdfs = 0

for pdf_folder in sorted(PDF_ROOT.glob("publications_*")):
    n = len([f for f in pdf_folder.iterdir() if f.suffix.lower() == ".pdf"])
    print(f"{pdf_folder.name}: {n}")
    total_pdfs += n

print("\n==========================")
print(f"Total PDFs across folders: {total_pdfs}")
